In [ ]:

# ANALISIS DE MÉTRICAS DE DIVERSIDAD
# Este script aplica un conjunto de enfoques complementarios para caracterizar
# la diversidad alfa en tu repertorio: primero calcula los números de Hill 
# mediante alphaDiversity de alakazam ; luego incorpora métricas adicionales 
# con vegan como índices de diversidad y equidad para describir la distribución
# de abundancias; y finalmente utiliza ineq para estimar desigualdad clonal a través
# del índice de Gini y otras medidas de concentración. Al combinar estas diez métricas, 
# obtienes una visión integrada de la cantidad, equilibrio y desigualdad en la arquitectura 
# clonal de tu repertorio.

In [1]:
# Paquetes y librerías
library(readr)
library(dplyr)
library(ggplot2)
library(viridisLite)
library(here)  # para rutas relativas
# install.packages("alakazam")
library(alakazam)
library(ineq)
library(vegan)
library(tidyr)
library(tibble)


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


here() starts at /Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/notebooks

Warning message:
"replacing previous import 'S4Arrays::makeNindexFromArrayViewport' by 'DelayedArray::makeNindexFromArrayViewport' when loading 'SummarizedExperiment'"


To cite the alakazam package in publications, please use:

  Gupta N, Vander Heiden J, Uduman M, Gadala-Maria D, Yaari G,
  Kleinstein S (2015). "Change-O: a toolkit for analyzing large-scale B
  cell immunoglobulin repertoire sequencing data." _Bioinformatics_,
  1-3. doi:10.1093/bioinformatics/btv359
  <https://doi.org/10.1093/bioinformatics/btv359>.

To cite the Ig-specific lineage reconstruction and diversity methods,
please use:

  Stern J, Yaari G, Vander Heiden J, Church G, Donahue W, Hintzen R,
  Huttner A, Laman J, Nagra R, Nylander

In [2]:

# Cargar tu archivo .tsv
archivo_clones <- "../data/output/repertorio_A_insilico_30_seqs_clone-pass.tsv"
clones <- read_tsv(archivo_clones)
# Agregar identificador de muestra (porque es simulado)
clones <- clones %>%
  mutate(sample_id = "repertorio_simulado")


# Contar secuencias por clon y muestra
clone_counts <- clones %>%
  group_by(sample_id, clone_id) %>%
  summarise(count = n(), .groups = "drop")

Rows: 30 Columns: 50
-- Column specification --------------------------------------------------------
Delimiter: "\t"
chr (20): sequence_id, sequence, v_call, d_call, j_call, sequence_alignment,...
dbl (25): junction_length, np1_length, np2_length, v_sequence_start, v_seque...
lgl  (5): rev_comp, productive, stop_codon, vj_in_frame, c_call

i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [4]:
hill <- alphaDiversity(
  data = clone_counts,
  clone = "clone_id",
  min_q = 0,
  max_q = 4,
  step_q = 1,
  nboot = 100,
  ci = 0.95
)

# Extraer la tabla
df <- hill@diversity

# Agregar columna con índices clásicos
df <- df %>%
  dplyr::mutate(
    indice_clasico = case_when(
      q == 0 ~ d,            # riqueza observada
      q == 1 ~ log(d),       # Shannon clásico H = ln(D1)
      q == 2 ~ 1/d,          # Simpson clásico D = 1/D2
      q == 3 ~ 1/(d^2),      # ∑ p_i^3 = 1/(D3^2)
      q == 4 ~ 1/(d^3)       # ∑ p_i^4 = 1/(D4^3)
    )
  )

print(df)
df_tbl <- as_tibble(df)

# A tibble: 5 x 10
# Groups:   group [1]
  group     q     d  d_sd d_lower d_upper     e e_lower e_upper indice_clasico
  <chr> <dbl> <dbl> <dbl>   <dbl>   <dbl> <dbl>   <dbl>   <dbl>          <dbl>
1 All       0  28.9 0.879    27.2    30.7 1       0.940    1.06     28.9      
2 All       1  28.6 1.14     26.3    30.8 0.988   0.910    1.07      3.35     
3 All       2  28.1 1.49     25.2    31.0 0.971   0.870    1.07      0.0356   
4 All       3  27.4 1.90     23.7    31.2 0.948   0.819    1.08      0.00133  
5 All       4  26.6 2.36     22.0    31.3 0.920   0.760    1.08      0.0000530


In [5]:
hill_numbers <- function(clones_df){
    clone_counts <- clones_df %>%
        group_by(sample_id, clone_id) %>%
        summarise(count = n(), .groups = "drop")
    hill <- alphaDiversity(
        data = clone_counts,
        clone = "clone_id",
        min_q = 0,
        max_q = 4,
        step_q = 1,
        nboot = 100,
        ci = 0.95
    )
    return(hill@diversity)
}
rep_hill_numbers <- hill_numbers(clones)
rep_hill_numbers

group,q,d,d_sd,d_lower,d_upper,e,e_lower,e_upper
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
All,0,28.90000,1.087115,26.76929,31.03071,1.0000000,0.9262732,1.073727
All,1,28.54387,1.405464,25.78921,31.29852,0.9876770,0.8923601,1.082994
All,2,28.05677,1.809550,24.51012,31.60342,0.9708226,0.8481010,1.093544
All,3,27.42717,2.279657,22.95912,31.89522,0.9490370,0.7944333,1.103641
All,4,26.67508,2.777122,21.23202,32.11814,0.9230130,0.7346719,1.111354


In [6]:
richness <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 0) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

richness(rep_hill_numbers)

[1] 28.9

In [10]:
 q1 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 1) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

q1(rep_hill_numbers)

[1] 28.54387

In [9]:
shannon <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 1) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    
    return(log(metric_value))
} 

shannon(rep_hill_numbers)

[1] 3.351442

In [14]:
 q2 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 2) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

q2(rep_hill_numbers)

[1] 28.05677

In [38]:
simpson <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 2) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    
    return(1/(metric_value))
} 

simpson(rep_hill_numbers)

[1] 0.03564202

In [16]:
 q3 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 3) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

q3(rep_hill_numbers)

[1] 27.42717

In [17]:
d3 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 1) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    
    return(1/(metric_value)^2)
} 

d3(rep_hill_numbers)

[1] 0.001227367

In [19]:
 q4 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 4) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

q4(rep_hill_numbers)

[1] 26.67508

In [39]:
d4 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 4) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    
    return(1/(metric_value)^3)
} 

d4(rep_hill_numbers)

[1] 5.268452e-05

In [27]:
# MÉTRICA CHAO1 y ACE PAQUETE VEGAN
metricas_chao1ace <- clone_counts %>%
  group_by(sample_id) %>% 
  summarise(
    chao1 = estimateR(count)["S.chao1"],
    ace   = estimateR(count)["S.ACE"]
  )

print(metricas_chao1ace)

# A tibble: 1 x 3
  sample_id           chao1   ace
  <chr>               <dbl> <dbl>
1 repertorio_simulado   465   NaN


In [49]:


chao1 <- function(clones_df){
  clone_counts <- clones_df %>%
    group_by(sample_id, clone_id) %>%
    summarise(count = n(), .groups = "drop")
  
  metricas_chao1 <- clone_counts %>%
    group_by(sample_id) %>%
    summarise(
      chao1 = as.numeric(vegan::estimateR(count)["S.chao1"]),
      .groups = "drop"
    )%>%
    dplyr::pull(chao1)
  
  return(metricas_chao1[1])
}

# Ejecutar
rep_chao1 <- chao1(clones)
print(rep_chao1)


[1] 465


In [54]:
ace <- function(clones_df){
  clone_counts <- clones_df %>%
    group_by(sample_id, clone_id) %>%
    summarise(count = n(), .groups = "drop")
  
  metricas_ace <- clone_counts %>%
    group_by(sample_id) %>%
    summarise(
      ace = as.numeric(vegan::estimateR(count)["S.ACE"]),
      .groups = "drop"
    )%>%
    dplyr::pull(ace)
  
  return(metricas_ace[1])
}

# Ejecutar
rep_ace <- ace(clones)
print(rep_ace)

[1] NaN


In [56]:
# MÉTRICA GINI PAQUETE INEQ
calc_gini <- function(df) {
  ineq::ineq(df$count, type = "Gini")
}
gini_result <- clone_counts %>%
  group_by(sample_id) %>%
  summarise(
    gini = calc_gini(cur_data())
  )

print(gini_result)

Warning message:
"There was 1 warning in `summarise()`.
i In argument: `gini = calc_gini(cur_data())`.
i In group 1: `sample_id = "repertorio_simulado"`.
Caused by warning:
! `cur_data()` was deprecated in dplyr 1.1.0.
i Please use `pick()` instead."


# A tibble: 1 x 2
  sample_id            gini
  <chr>               <dbl>
1 repertorio_simulado     0


In [61]:
gini <- function (clones_df){
    clone_counts <- clones_df %>%
        group_by(sample_id, clone_id) %>%
        summarise(count = n(), .groups = "drop")
    metrica_gini <- ineq::ineq(clone_counts$count, type = "Gini")
    return(metrica_gini)
}
gini(clones)

[1] 0

In [69]:
metricas_diversidad <- c(richness= richness(rep_hill_numbers),q1= q1(rep_hill_numbers),shannon= shannon(rep_hill_numbers), q2= q2(rep_hill_numbers), simpson= simpson(rep_hill_numbers), q3= q3(rep_hill_numbers), 
d3= d3(rep_hill_numbers), q4= q4(rep_hill_numbers), d4= d4(rep_hill_numbers), chao1= chao1(clones), gini= gini(clones))
metricas_diversidad

richness           q1      shannon           q2      simpson           q3 
2.890000e+01 2.854387e+01 3.351442e+00 2.805677e+01 3.564202e-02 2.742717e+01 
          d3           q4           d4        chao1         gini 
1.227367e-03 2.667508e+01 5.268452e-05 4.650000e+02 0.000000e+00

In [75]:
tabla_diversidad <- bind_rows(metricas_diversidad)
tabla_diversidad$sample_id <- "A_30seq"
tabla_diversidad

richness,q1,shannon,q2,simpson,q3,d3,q4,d4,chao1,gini,sample_id
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
28.9,28.54387,3.351442,28.05677,0.03564202,27.42717,0.001227367,26.67508,5.268452e-05,465,0,A_30seq


In [77]:
readr::write_tsv(tabla_diversidad, "../results/diversity_metrics/diversity_A_30seqs.tsv")